# 11 - Dashboard Data Preparation

## Objective

This notebook prepares dashboard-ready datasets for the Streamlit application and optional FastAPI layer by aggregating historical metrics, scored flights, SHAP insights, statistical validation, and prioritization results into reusable Delta tables.

## Notebook 07–10 Compatibility

The dashboard continues to read the established predictions, prioritization, SHAP, and evaluation tables. These tables are now produced from the dynamically selected standard-Python model bundle, so no dashboard-facing schema change is required. Model labels and metrics must be read from the persisted artifacts rather than assumed to be Logistic Regression.


#### Load project configuration

In [0]:
import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

from config import project_config as cfg

print("Project configuration loaded successfully.")
print(f"Dashboard table: {cfg.DASHBOARD_TABLE}")
print(f"Explorer table: {cfg.DASHBOARD_EXPLORER_TABLE}")
print(f"Insights table: {cfg.DASHBOARD_INSIGHTS_TABLE}")


Project configuration loaded successfully.
Dashboard table: workspace.default.flight_dashboard
Explorer table: workspace.default.flight_dashboard_explorer
Insights table: workspace.default.flight_dashboard_insights


#### Load source datasets

The dashboard assembly combines cleaned operational history, model predictions, SHAP artifacts, statistical validation outputs, and prioritization evaluation results.

In [0]:
from __future__ import annotations

import json

from pyspark.sql import functions as F


def require_table(table_name: str) -> None:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the upstream notebooks before continuing."
        )


def require_file(file_path: str) -> None:
    try:
        dbutils.fs.head(file_path, 1)
    except Exception as exc:
        raise RuntimeError(
            f"Required file '{file_path}' was not found. "
            "Run the upstream notebooks before continuing."
        ) from exc


required_tables = [
    cfg.CLEAN_TABLE,
    cfg.PREDICTIONS_TABLE,
    cfg.STATISTICAL_RESULTS_TABLE,
    cfg.SHAP_GLOBAL_IMPORTANCE_TABLE,
    cfg.SHAP_DIRECTION_EFFECTS_TABLE,
    cfg.PRIORITIZATION_EVALUATION_TABLE,
]

for table_name in required_tables:
    require_table(table_name)

require_file(cfg.SELECTED_MODEL_METRICS_PATH)

df_clean = spark.table(cfg.CLEAN_TABLE)
df_predictions = spark.table(cfg.PREDICTIONS_TABLE)
df_statistical = spark.table(cfg.STATISTICAL_RESULTS_TABLE)
df_shap_global = spark.table(cfg.SHAP_GLOBAL_IMPORTANCE_TABLE)
df_shap_direction = spark.table(cfg.SHAP_DIRECTION_EFFECTS_TABLE)
df_prioritization_eval = spark.table(cfg.PRIORITIZATION_EVALUATION_TABLE)
model_metrics = json.loads(
    dbutils.fs.head(cfg.SELECTED_MODEL_METRICS_PATH, 1000000)
)

print("Dashboard source datasets loaded successfully.")
print(f"Predictions rows: {df_predictions.count():,}")
print(f"SHAP features: {df_shap_global.count():,}")


[Truncated to first 1 bytes]
Dashboard source datasets loaded successfully.
Predictions rows: 1,159,898
SHAP features: 1,428


#### Build overview and historical dashboard metrics

In [0]:
import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

import pandas as pd

from utils.dashboard_preparation import (
    MONTH_LABELS,
    build_delay_cause_records,
    build_model_metric_records,
    build_monthly_trend_records,
    build_overview_kpi_records,
    build_research_validation_records,
    combine_dashboard_records,
    serialize_dashboard_metadata,
)


eligible_flights = df_clean.filter(
    (F.col(cfg.CANCELLED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
    & (F.col(cfg.DIVERTED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
)


eligible_overview_row = eligible_flights.agg(
    F.count("*").alias("total_flights"),
    (
        F.avg(cfg.TARGET_COLUMN) * 100
    ).alias("delay_rate"),
    F.avg(cfg.ARRIVAL_DELAY_COLUMN).alias(
        "average_arrival_delay"
    ),
).collect()[0]


cancellation_row = df_clean.agg(
    (
        F.avg(
            F.col(cfg.CANCELLED_COLUMN).cast("double")
        ) * 100
    ).alias("cancellation_rate")
).collect()[0]

overview_records = build_overview_kpi_records(
    total_flights=int(
        eligible_overview_row["total_flights"]
    ),
    delay_rate=float(
        eligible_overview_row["delay_rate"] or 0.0
    ),
    average_arrival_delay=float(
        eligible_overview_row["average_arrival_delay"] or 0.0
    ),
    cancellation_rate=float(
        cancellation_row["cancellation_rate"] or 0.0
    ),
)

monthly_trend_pdf = (
    eligible_flights.groupBy(cfg.MONTH_COLUMN)
    .agg((F.avg(cfg.TARGET_COLUMN) * 100).alias("delay_rate"))
    .orderBy(cfg.MONTH_COLUMN)
    .toPandas()
)
monthly_trend_pdf["month_label"] = monthly_trend_pdf[cfg.MONTH_COLUMN].map(MONTH_LABELS)
monthly_trend_pdf["month_number"] = monthly_trend_pdf[cfg.MONTH_COLUMN]
monthly_records = build_monthly_trend_records(monthly_trend_pdf)

cause_expressions = [
    F.sum(F.col(column_name)).alias(column_name)
    for column_name in cfg.DELAY_CAUSE_COLUMNS
]
cause_totals = eligible_flights.agg(*cause_expressions).collect()[0].asDict()
total_delay_minutes = float(sum(cause_totals.values())) or 1.0
cause_rows = []
for column_name, label in cfg.DELAY_CAUSE_LABELS.items():
    cause_rows.append(
        {
            "cause": label,
            "percentage": round((cause_totals[column_name] / total_delay_minutes) * 100, 2),
        }
    )
cause_records = build_delay_cause_records(pd.DataFrame(cause_rows))

display(spark.createDataFrame(overview_records))
display(spark.createDataFrame(monthly_records))


section,metric_name,metric_value,metric_text,dimension_1,dimension_2,sort_order
overview_kpi,total_flights,6879484.0,6879484,null,null,1
overview_kpi,avg_delay_rate,22.307457943066662,22.3%,null,null,2
overview_kpi,avg_arr_delay,8.504504407598011,8.5 min,null,null,3
overview_kpi,cancel_rate,1.4693175206073796,1.47%,null,null,4


section,metric_name,metric_value,metric_text,dimension_1,dimension_2,sort_order
monthly_trend,delay_rate,18.78916803409737,Jan,Jan,null,1
monthly_trend,delay_rate,20.76676415375567,Feb,Feb,null,2
monthly_trend,delay_rate,19.590377189976042,Mar,Mar,null,3
monthly_trend,delay_rate,19.663856818929258,Apr,Apr,null,4
monthly_trend,delay_rate,23.601763128917923,May,May,null,5
monthly_trend,delay_rate,28.259034617129743,Jun,Jun,null,6
monthly_trend,delay_rate,28.88525173340557,Jul,Jul,null,7
monthly_trend,delay_rate,22.555593170667624,Aug,Aug,null,8
monthly_trend,delay_rate,16.631621555787994,Sep,Sep,null,9
monthly_trend,delay_rate,20.321990790764165,Oct,Oct,null,10


In [0]:
df_clean.groupBy(cfg.CANCELLED_COLUMN).count().show()

+---------+-------+
|CANCELLED|  count|
+---------+-------+
|        1| 102876|
|        0|6898742|
+---------+-------+



#### Build explorer, insights, and research-validation datasets

In [0]:
predictions_pdf = df_predictions.toPandas()

explorer_pdf = predictions_pdf.assign(
    Flight=predictions_pdf["flight_label"],
    Carrier=predictions_pdf["airline_code"],
    Origin=predictions_pdf["origin_airport"],
    Destination=predictions_pdf["destination_airport"],
    SchedDep=predictions_pdf["scheduled_departure_text"],
    DepartureWindow=predictions_pdf["departure_window"],
    DepTime=predictions_pdf["scheduled_departure_text"],
    DelayProb=predictions_pdf["delay_probability"],
    DelayProbPct=(predictions_pdf["delay_probability"] * 100).round(1).astype(str) + "%",
    RiskTier=predictions_pdf["risk_level"],
    Status=predictions_pdf["risk_level"],
    Month=predictions_pdf["month_number"].map(MONTH_LABELS),
    ShapMainDriver=predictions_pdf["shap_main_driver"],
)

insights_global_pdf = df_shap_global.toPandas().rename(
    columns={"Feature": "feature", "MeanAbsSHAP": "importance"}
)
insights_direction_pdf = df_shap_direction.toPandas()

statistical_pdf = df_statistical.toPandas()
statistical_records = build_research_validation_records(
    statistical_pdf.assign(
        metric_name=statistical_pdf["research_question"],
        metric_value=statistical_pdf["p_value"],
        metric_text=statistical_pdf["decision"],
        dimension_1=statistical_pdf["factor"],
        dimension_2=statistical_pdf["test_name"],
    )
)

prioritization_eval_pdf = df_prioritization_eval.toPandas()
rq4_records = build_research_validation_records(
    prioritization_eval_pdf.assign(
        metric_name="RQ4",
        metric_value=prioritization_eval_pdf["delay_recall"],
        metric_text=prioritization_eval_pdf["strategy"],
        dimension_1=prioritization_eval_pdf["capacity_k"].astype(str),
        dimension_2=prioritization_eval_pdf["strategy"],
    )
)

model_metric_records = build_model_metric_records(model_metrics)
dashboard_records_pdf = combine_dashboard_records(
    [
        overview_records,
        monthly_records,
        cause_records,
        statistical_records,
        rq4_records,
        model_metric_records,
    ]
)

display(spark.createDataFrame(dashboard_records_pdf).limit(20))
display(spark.createDataFrame(explorer_pdf).select(
    "Flight", "Carrier", "Origin", "Destination", "DelayProb", "RiskTier", "Month"
).limit(10))
display(spark.createDataFrame(insights_global_pdf).head(10))


section,metric_name,metric_value,metric_text,dimension_1,dimension_2,sort_order
overview_kpi,total_flights,6879484.0,6879484,null,null,1
overview_kpi,avg_delay_rate,22.307457943066662,22.3%,null,null,2
overview_kpi,avg_arr_delay,8.504504407598011,8.5 min,null,null,3
overview_kpi,cancel_rate,1.4693175206073796,1.47%,null,null,4
monthly_trend,delay_rate,18.78916803409737,Jan,Jan,null,1
monthly_trend,delay_rate,20.76676415375567,Feb,Feb,null,2
monthly_trend,delay_rate,19.590377189976042,Mar,Mar,null,3
monthly_trend,delay_rate,19.663856818929258,Apr,Apr,null,4
monthly_trend,delay_rate,23.601763128917923,May,May,null,5
monthly_trend,delay_rate,28.259034617129743,Jun,Jun,null,6


Flight,Carrier,Origin,Destination,DelayProb,RiskTier,Month
UA-1164-SFO-LAX-1050,UA,SFO,LAX,0.23284463822379708,LOW,Oct
UA-1202-DTW-ORD-1413,UA,DTW,ORD,0.42181778738716225,MEDIUM,Oct
UA-1219-DEN-MTJ-1142,UA,DEN,MTJ,0.3397713975823772,MEDIUM,Oct
UA-1224-DEN-ORD-1645,UA,DEN,ORD,0.61749103451829,HIGH,Oct
UA-1234-CHS-EWR-600,UA,CHS,EWR,0.2139220440658004,LOW,Oct
UA-1241-DEN-CID-1056,UA,DEN,CID,0.2801926906990505,LOW,Oct
UA-1260-DEN-ORD-1615,UA,DEN,ORD,0.6040332244523647,HIGH,Oct
UA-1278-ORD-LGA-1700,UA,ORD,LGA,0.5506178040663903,MEDIUM,Oct
UA-1284-CLE-IAD-1035,UA,CLE,IAD,0.18393339947680554,LOW,Oct
UA-1286-SFO-DFW-1620,UA,SFO,DFW,0.48940281327377977,MEDIUM,Oct


feature,importance
FLIGHT DISTANCE CATEGORY Medium,3.459573351959342
FLIGHT DISTANCE CATEGORY Short,3.0508802878142935
SEASON Spring,2.346150158197219
SEASON Fall,2.3114538585842173
Historical Route Delay Rate,2.181764660528529
TIME OF DAY Morning,2.1405470650220177
Departure Hour,1.9920980914542608
Scheduled Departure Time,1.9917802958548902
SEASON Winter,1.9805007700686237
Month,1.6179558084581964


#### Save dashboard-ready datasets

In [0]:
dashboard_records_df = spark.createDataFrame(dashboard_records_pdf)
explorer_df = spark.createDataFrame(explorer_pdf)
insights_df = spark.createDataFrame(insights_global_pdf)

(
    dashboard_records_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(cfg.DASHBOARD_DELTA_PATH)
)
(
    dashboard_records_df.writeTo(cfg.DASHBOARD_TABLE).using("delta").createOrReplace()
)

(
    explorer_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(cfg.DASHBOARD_EXPLORER_PATH)
)
(
    explorer_df.writeTo(cfg.DASHBOARD_EXPLORER_TABLE).using("delta").createOrReplace()
)

(
    insights_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(cfg.DASHBOARD_INSIGHTS_PATH)
)
(
    insights_df.writeTo(cfg.DASHBOARD_INSIGHTS_TABLE).using("delta").createOrReplace()
)

dashboard_metadata = {
    "dashboard_table": cfg.DASHBOARD_TABLE,
    "explorer_table": cfg.DASHBOARD_EXPLORER_TABLE,
    "insights_table": cfg.DASHBOARD_INSIGHTS_TABLE,
    "predictions_table": cfg.PREDICTIONS_TABLE,
    "statistical_results_table": cfg.STATISTICAL_RESULTS_TABLE,
    "prioritization_evaluation_table": cfg.PRIORITIZATION_EVALUATION_TABLE,
    "research_questions": cfg.RESEARCH_QUESTIONS,
    "default_capacity_k": cfg.DEFAULT_CAPACITY_K,
}

dbutils.fs.put(
    cfg.DASHBOARD_METADATA_PATH,
    serialize_dashboard_metadata(dashboard_metadata),
    overwrite=True,
)

print("Dashboard datasets saved successfully.")
print(f"Dashboard table: {cfg.DASHBOARD_TABLE}")
print(f"Explorer table: {cfg.DASHBOARD_EXPLORER_TABLE}")
print(f"Insights table: {cfg.DASHBOARD_INSIGHTS_TABLE}")
print(f"Metadata file: {cfg.DASHBOARD_METADATA_PATH}")


Wrote 1096 bytes.
Dashboard datasets saved successfully.
Dashboard table: workspace.default.flight_dashboard
Explorer table: workspace.default.flight_dashboard_explorer
Insights table: workspace.default.flight_dashboard_insights
Metadata file: /Volumes/workspace/default/flight_delay_capstone/models/dashboard_metadata.json
